# Lesson 6: Full Tabular Pipeline — MLP with Embeddings

In Lesson 5 we took apart the neural network engine — forward passes, loss functions, backpropagation, optimizers. We understood the mechanics. Now we put that engine into a real car and drive it.

This notebook walks through the **complete pipeline** for training a neural network on tabular data: exploring the data, cleaning it up, encoding features, building a model with embeddings, training it, tuning hyperparameters, and evaluating how well it actually works.

We'll use the **Adult Income** dataset — census data where we predict whether someone earns more than $50K/year. It has a good mix of categorical and numerical features, some messy real-world data issues, and a clear binary target.

## Pipeline Overview

Here's the road map. Each section builds on the previous one:

**A1. Explore the Data** — Understand what we're working with. What do the features mean? Any issues?

**A2. Preprocess** — Handle missing values, encode categories as numbers, scale features, build PyTorch DataLoaders

**A3. Build the Model** — Design an MLP with embedding layers for categorical features

**A4. Train** — Loss function, optimizer, training loop with validation

**A5. Tune Hyperparameters** — Experiment with different settings and see how they affect training

**A6. Evaluate** — Measure real performance with proper metrics

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score, RocCurveDisplay

## A1. Explore the Data

The **Adult Income** dataset comes from the 1994 US Census. Each row represents a person, and the target tells us whether they earn more than \$50K per year. It's a classic binary classification problem — given someone's age, education, occupation, and other census features, can we predict their income bracket?

The dataset has no header row, so we'll define column names ourselves.

In [ ]:
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

df = pd.read_csv(
    '/data/datasets/adult_income/adult.data',
    names=column_names,
    skipinitialspace=True
)

df.shape
df.head()

In [ ]:
df.dtypes

We've got a mix of types: `object` columns are categorical (text), `int64` columns are numerical. Let's dig into what each feature actually represents.

**Numerical features:**
- `age` — person's age
- `fnlwgt` — a Census sampling weight (how many people this row represents in the population). Not useful for prediction.
- `education_num` — years of education as a number (1 = preschool, 16 = doctorate)
- `capital_gain` / `capital_loss` — investment income/losses in the past year
- `hours_per_week` — typical working hours

**Categorical features:**
- `workclass` — type of employer (Private, Government, Self-employed, etc.)
- `education` — education level as text (Bachelors, HS-grad, Masters, etc.)
- `marital_status` — married, divorced, never-married, etc.
- `occupation` — what they do (Tech-support, Sales, Exec-managerial, etc.)
- `relationship` — family role (Husband, Wife, Not-in-family, etc.)
- `race`, `sex` — demographic info
- `native_country` — country of origin

In [ ]:
df.describe()

### Missing Values

Real datasets are messy. Let's check if anything is missing. This dataset uses `"?"` instead of proper NaN values for missing data — a common pattern in older datasets.

In [ ]:
for col in df.columns:
    missing_count = (df[col] == '?').sum()
    if missing_count > 0:
        print(f"{col}: {missing_count} missing ({missing_count / len(df) * 100:.1f}%)")

Three columns have missing values: `workclass`, `occupation`, and `native_country`. The `workclass` and `occupation` missings likely overlap — if someone didn't report their employer type, they probably didn't report their job either.

We have a few options for handling these:
1. **Drop rows** with missing values — simple, loses some data
2. **Treat "?" as its own category** — model learns "unknown" as a meaningful group
3. **Impute** — fill with most common value or use a model to predict

With ~2,400 affected rows out of 32,561 (about 7%), dropping them is the simplest choice and still leaves us with 30K+ rows. Let's go with that.

In [ ]:
rows_before = len(df)
df = df[(df != '?').all(axis=1)]
rows_after = len(df)
print(f"Dropped {rows_before - rows_after} rows with missing values")
print(f"Remaining: {rows_after} rows")

### Redundant and Irrelevant Features

Before we go further, let's clean house. Two columns need to go:

In [ ]:
# education and education_num encode the same information
df.groupby('education')['education_num'].first().sort_values()

`education` (text) and `education_num` (number) are a 1-to-1 mapping. Every "Bachelors" is a 13, every "HS-grad" is a 9. Keeping both would be feeding the model the same information twice. Since `education_num` is already a clean numerical feature, we'll drop the text version.

`fnlwgt` is a Census sampling weight — it tells the Census Bureau how many people a given row represents in the US population. It's metadata about the survey methodology, not a feature about the person. Not useful for predicting income.

In [ ]:
df = df.drop(columns=['education', 'fnlwgt'])
df.shape

### Examining Categorical Features

Let's see what categories each feature has. This matters because the number of unique values per column determines the size of our embedding tables later.

In [ ]:
cat_cols_explore = ['workclass', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']

for col in cat_cols_explore:
    print(f"\n{col} ({df[col].nunique()} unique):")
    print(df[col].value_counts().to_string())

A few things stand out:

- `native_country` has 41 categories but ~90% is "United-States". Lots of categories with very few examples. The embedding layer will handle this fine — rare categories just won't get well-trained embeddings.
- `occupation` has 14 categories with decent spread — good for embeddings.
- `sex` and `race` have low cardinality (2 and 5 respectively) — small embedding tables.

### Target Variable

In [ ]:
df['income'].value_counts()
df['income'].value_counts(normalize=True)

About 75% earn ≤\$50K and 25% earn >\$50K. There's some class imbalance here — more "no" than "yes" — but it's moderate. A model that just guesses "≤50K" for everyone would get ~75% accuracy, so we need to beat that baseline clearly. We'll handle this with a weighted loss function during training.

## A2. Preprocessing

Our data exploration gave us a clean understanding of the dataset. Now we transform it into something a neural network can consume. The steps:

1. Encode the target (text → number)
2. Split into train/validation sets
3. Label encode categorical features
4. Scale numerical features
5. Package into PyTorch DataLoaders

### Encode the Target

In [ ]:
df['income'] = (df['income'] == '>50K').astype(int)
df['income'].value_counts()

### Train/Validation Split

We split the data *before* any encoding or scaling. This prevents data leakage — we don't want statistics from the validation set influencing how we transform the training data.

We use a **stratified** split to maintain the same class proportions (75/25) in both sets.

In [ ]:
X = df.drop(columns=['income'])
y = df['income']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} rows")
print(f"Val:   {len(X_val)} rows")
print(f"\nTrain class balance: {y_train.mean():.1%} positive")
print(f"Val class balance:   {y_val.mean():.1%} positive")

### Define Feature Groups

We need to treat categorical and numerical features differently:
- **Categorical** → label encode (convert text to integers), then feed into embedding layers
- **Numerical** → scale to similar ranges, then feed directly into the network

In [ ]:
cat_columns = ['workclass', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']
num_columns = ['age', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

print(f"Categorical features: {len(cat_columns)}")
print(f"Numerical features:  {len(num_columns)}")

### Label Encode Categoricals

Neural networks only understand numbers, not strings like "Private" or "Bachelors". Label encoding assigns each unique category an integer: Private → 0, Self-emp → 1, Government → 2, etc.

Later, our model will use these integers as lookup indices into **embedding tables** — learnable vectors that represent each category. But for now, we just need the integers.

Important: we **fit** the encoder on training data only, then **apply** it to validation data. This mirrors how it would work in production — you can't peek at future data to decide your encoding.

In [ ]:
label_encoders = {}

for col in cat_columns:
    encoder = LabelEncoder()
    X_train[col] = encoder.fit_transform(X_train[col])
    X_val[col] = encoder.transform(X_val[col])
    label_encoders[col] = encoder
    print(f"{col}: {len(encoder.classes_)} categories → {list(encoder.classes_[:5])}{'...' if len(encoder.classes_) > 5 else ''}")

### Scale Numerical Features

Neural networks multiply inputs by weights. If one feature ranges from 0 to 99,999 (`capital_gain`) and another from 1 to 16 (`education_num`), the large-scale feature dominates the learning — the network spends most of its energy adjusting weights for the big numbers.

`StandardScaler` transforms each feature to have mean=0 and standard deviation=1. After scaling, all features live in roughly the same range, giving each one a fair shot at influencing the model.

Again — fit on train, apply to both.

In [ ]:
scaler = StandardScaler()
X_train[num_columns] = scaler.fit_transform(X_train[num_columns])
X_val[num_columns] = scaler.transform(X_val[num_columns])

X_train[num_columns].describe().round(2)

### PyTorch Dataset and DataLoader

Pandas DataFrames are great for exploration, but PyTorch needs tensors. We'll create a custom `Dataset` class that:
- Stores categorical features as `torch.long` (integer type — needed for embedding lookups)
- Stores numerical features as `torch.float32` (decimal type — needed for math)
- Returns batches of (categoricals, numericals, labels)

Then we wrap it in a `DataLoader` that handles batching and shuffling.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y, cat_columns, num_columns):
        self.cats = torch.tensor(X[cat_columns].values, dtype=torch.long)
        self.nums = torch.tensor(X[num_columns].values, dtype=torch.float32)
        self.labels = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.cats[idx], self.nums[idx], self.labels[idx]

In [ ]:
train_dataset = TabularDataset(X_train, y_train, cat_columns, num_columns)
val_dataset = TabularDataset(X_val, y_val, cat_columns, num_columns)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

In [ ]:
# Quick check — grab one batch and inspect shapes
cats_batch, nums_batch, labels_batch = next(iter(train_loader))
print(f"Categorical batch: {cats_batch.shape}")  # [256, 7]
print(f"Numerical batch:   {nums_batch.shape}")  # [256, 5]
print(f"Labels batch:      {labels_batch.shape}") # [256]

Each batch gives us 256 rows. The categorical tensor has 7 columns (one integer per categorical feature), the numerical tensor has 5 columns, and labels are a flat vector of 0s and 1s.

Our data pipeline is complete. Everything from raw CSV to model-ready tensors flowing through a DataLoader. Now we build the model.

## A3. Build the Model

Our model needs to handle two types of input: categorical features (encoded as integers) and numerical features (scaled floats). We can't just throw both into the same layer — an integer like `occupation=7` doesn't have numerical meaning the way `age=35` does.

The solution: **embedding layers** for categoricals, then concatenate everything into a standard MLP.

### What Are Embeddings?

Think of an embedding table as a lookup dictionary. For each categorical feature, we create a small table where each category maps to a learnable vector of numbers.

For example, the `occupation` column has 14 categories. We create a table with 14 rows and (say) 7 columns. When the model sees `occupation=5` ("Sales"), it looks up row 5 and gets 7 numbers. These numbers start random but get refined during training — the model learns what representation of "Sales" is useful for predicting income.

This is more powerful than one-hot encoding because:
- The embedding dimensions capture **relationships** (similar occupations get similar vectors)
- It's more **compact** (7 numbers instead of 14 binary columns)
- The representations are **learned**, not hand-crafted

### Embedding Sizes

For each categorical column, we need to decide: how many dimensions should its embedding have? A common rule of thumb: `min(50, (num_categories + 1) // 2)`. More categories get more dimensions, but we cap at 50.

In [ ]:
embedding_sizes = []
for col in cat_columns:
    num_categories = len(label_encoders[col].classes_)
    embedding_dim = min(50, (num_categories + 1) // 2)
    embedding_sizes.append((num_categories, embedding_dim))
    print(f"{col}: {num_categories} categories → {embedding_dim}-dim embedding")

total_embedding_dim = sum(dim for _, dim in embedding_sizes)
print(f"\nTotal embedding dimensions: {total_embedding_dim}")
print(f"Plus {len(num_columns)} numerical features")
print(f"Total input to MLP: {total_embedding_dim + len(num_columns)}")

### The Architecture

Here's the full picture of how data flows through our model:

1. **Categorical inputs** (7 integers) → each one looked up in its embedding table → 7 vectors of varying lengths
2. **All embeddings + numerical features** → concatenated into one long vector
3. **MLP** → two hidden layers with ReLU activations → single output logit

The output is a **logit** — an unbounded number where positive means "likely >50K" and negative means "likely ≤50K". We'll convert this to a probability with sigmoid later.

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, embedding_sizes, num_numerical_features, hidden_size=256, dropout_rate=0.0):
        super().__init__()

        # One embedding table per categorical feature
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_categories, embedding_dim)
            for num_categories, embedding_dim in embedding_sizes
        ])

        # Total input size = all embedding dims + numerical features
        total_embedding_dim = sum(dim for _, dim in embedding_sizes)
        input_size = total_embedding_dim + num_numerical_features

        # MLP layers
        self.layers = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, categorical_features, numerical_features):
        # Look up each categorical feature in its embedding table
        embedded = [emb(categorical_features[:, i]) for i, emb in enumerate(self.embeddings)]

        # Concatenate all embeddings + numerical features into one vector
        combined = torch.cat(embedded + [numerical_features], dim=1)

        # Pass through MLP
        return self.layers(combined)

In [ ]:
# Build the model with default settings (we'll tune later)
model = NeuralNetwork(
    embedding_sizes=embedding_sizes,
    num_numerical_features=len(num_columns),
    hidden_size=256,
    dropout_rate=0.0
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(model)

The model takes in 7 categorical integers and 5 numerical floats, passes them through embedding lookups and two hidden layers, and outputs a single logit. That's our architecture. Now we train it.

## A4. Training

Training means showing the model data over and over, computing how wrong it is (loss), and nudging the weights in the right direction (optimizer). We need three ingredients:

1. A **loss function** that measures prediction error
2. An **optimizer** that updates the weights
3. A **training loop** that puts it all together

### Loss Function

For binary classification (two classes: >50K or ≤50K), we use `BCEWithLogitsLoss`. This combines a sigmoid activation with binary cross-entropy loss in one step — it takes the model's raw logit output and computes how far off it is from the true label.

Remember our class distribution: ~75% earn ≤50K, ~25% earn >50K. Without any adjustment, the model would see three times as many "low income" examples. It could learn a lazy shortcut: just predict "≤50K" for everyone and be right 75% of the time.

To fix this, we use `pos_weight` — it tells the loss function to penalize mistakes on the positive class (>50K) more heavily. The weight is the ratio of negative to positive examples.

In [ ]:
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()])
print(f"Positive class weight: {pos_weight.item():.2f}")
print(f"(meaning: a mistake on >50K costs {pos_weight.item():.1f}x more than a mistake on <=50K)")

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

### Optimizer

The optimizer's job is to update the model's weights after each batch. It looks at the gradients (which direction reduces the loss) and takes a step in that direction.

We'll use **Adam** — it adapts the learning rate for each parameter individually, which typically works better out of the box than plain SGD. The `lr` (learning rate) controls how big each step is.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### The Training Loop

Each **epoch** is one full pass through the training data. Within each epoch:
- Training data is split into **batches** (256 rows each)
- For each batch: forward pass → compute loss → backward pass → update weights
- After all training batches: evaluate on validation data (no weight updates)

The validation loss tells us if the model is learning general patterns (both losses drop) or just memorizing training data (train drops, val rises = overfitting).

In [ ]:
InteractiveShell.ast_node_interactivity = "last_expr"

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = model.to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

epochs = 20

for epoch in range(epochs):
    # --- Training phase ---
    model.train()
    train_loss_total = 0

    for cats, nums, labels in train_loader:
        cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)

        logits = model(cats, nums).squeeze()
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_total += loss.item()

    # --- Validation phase ---
    model.eval()
    val_loss_total = 0

    with torch.no_grad():
        for cats, nums, labels in val_loader:
            cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
            logits = model(cats, nums).squeeze()
            val_loss_total += loss_fn(logits, labels).item()

    # --- Print progress ---
    avg_train = train_loss_total / len(train_loader)
    avg_val = val_loss_total / len(val_loader)
    print(f"Epoch {epoch+1:2d}/{epochs}: train_loss={avg_train:.4f}, val_loss={avg_val:.4f}")

Watch the two loss values. If train loss keeps dropping but val loss starts climbing, the model is **overfitting** — memorizing training examples instead of learning general patterns. We'll address this in the hyperparameter tuning section.

## A5. Hyperparameter Tuning

The training loop works, but the model's behavior depends on choices we made: how many neurons, what learning rate, whether we use dropout. These are **hyperparameters** — settings we choose before training, not values the model learns.

Let's experiment with different configurations and see how they affect training. We'll try three approaches, each teaching something different.

### What Is Regularization?

Imagine studying for an exam. You could memorize every practice problem word for word — you'd ace those exact questions but fail anything slightly different. Or you could learn the underlying concepts — you might not get every practice problem perfect, but you'd handle new questions well.

Neural networks face the same choice. A big, powerful network can memorize the training data perfectly but fail on new data (overfitting). **Regularization** is a set of techniques that push the model toward learning general patterns instead of memorizing specifics:

- **Dropout**: Randomly turns off neurons during training. Forces the network to not rely on any single neuron — like studying with random pages of your notes removed.
- **Weight decay**: Adds a small penalty for large weights. Keeps the model from developing extreme, specialized connections.
- **Fewer neurons**: Less capacity to memorize. A smaller network *has* to learn efficient representations.
- **Lower learning rate**: Takes smaller steps. Less likely to overfit to noise in the data.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Attempt 1: Large Network, No Regularization

Let's start with an overfit-prone setup: 512 neurons, no dropout, relatively high learning rate. This should learn fast but likely memorize.

In [ ]:
# Attempt 1: 512 neurons, no dropout, lr=1e-3
model_1 = NeuralNetwork(embedding_sizes, len(num_columns), hidden_size=512, dropout_rate=0.0).to(device)
optimizer_1 = torch.optim.Adam(model_1.parameters(), lr=1e-3)
loss_fn_1 = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

for epoch in range(20):
    model_1.train()
    train_loss = 0
    for cats, nums, labels in train_loader:
        cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
        logits = model_1(cats, nums).squeeze()
        loss = loss_fn_1(logits, labels)
        optimizer_1.zero_grad()
        loss.backward()
        optimizer_1.step()
        train_loss += loss.item()

    model_1.eval()
    val_loss = 0
    with torch.no_grad():
        for cats, nums, labels in val_loader:
            cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
            val_loss += loss_fn_1(model_1(cats, nums).squeeze(), labels).item()

    print(f"Epoch {epoch+1:2d}: train={train_loss/len(train_loader):.4f}, val={val_loss/len(val_loader):.4f}")

**What to watch for**: Training loss likely drops quickly while validation loss either stagnates or starts climbing after a few epochs. The gap between them is the overfitting signal — the model is memorizing training patterns that don't generalize.

### Attempt 2: Small Network, Heavy Regularization

Now the opposite extreme: 128 neurons, high dropout (0.3), low learning rate. This should resist overfitting but might learn too slowly.

In [ ]:
# Attempt 2: 128 neurons, dropout=0.3, lr=3e-4, weight_decay=1e-4
model_2 = NeuralNetwork(embedding_sizes, len(num_columns), hidden_size=128, dropout_rate=0.3).to(device)
optimizer_2 = torch.optim.Adam(model_2.parameters(), lr=3e-4, weight_decay=1e-4)
loss_fn_2 = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

for epoch in range(30):
    model_2.train()
    train_loss = 0
    for cats, nums, labels in train_loader:
        cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
        logits = model_2(cats, nums).squeeze()
        loss = loss_fn_2(logits, labels)
        optimizer_2.zero_grad()
        loss.backward()
        optimizer_2.step()
        train_loss += loss.item()

    model_2.eval()
    val_loss = 0
    with torch.no_grad():
        for cats, nums, labels in val_loader:
            cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
            val_loss += loss_fn_2(model_2(cats, nums).squeeze(), labels).item()

    print(f"Epoch {epoch+1:2d}: train={train_loss/len(train_loader):.4f}, val={val_loss/len(val_loader):.4f}")

**What to watch for**: Both losses should drop together with little gap between them. The model is well-constrained but might not reach its full potential — if val loss is still improving at the end, we could train longer or reduce regularization.

### Attempt 3: Balanced Settings

A middle ground: 256 neurons, moderate dropout (0.15), medium learning rate. We'll also train for more epochs to see the full training curve.

In [ ]:
# Attempt 3: 256 neurons, dropout=0.15, lr=5e-4, weight_decay=1e-5
model_3 = NeuralNetwork(embedding_sizes, len(num_columns), hidden_size=256, dropout_rate=0.15).to(device)
optimizer_3 = torch.optim.Adam(model_3.parameters(), lr=5e-4, weight_decay=1e-5)
loss_fn_3 = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

for epoch in range(40):
    model_3.train()
    train_loss = 0
    for cats, nums, labels in train_loader:
        cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
        logits = model_3(cats, nums).squeeze()
        loss = loss_fn_3(logits, labels)
        optimizer_3.zero_grad()
        loss.backward()
        optimizer_3.step()
        train_loss += loss.item()

    model_3.eval()
    val_loss = 0
    with torch.no_grad():
        for cats, nums, labels in val_loader:
            cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
            val_loss += loss_fn_3(model_3(cats, nums).squeeze(), labels).item()

    print(f"Epoch {epoch+1:2d}: train={train_loss/len(train_loader):.4f}, val={val_loss/len(val_loader):.4f}")

**What to watch for**: Validation loss should improve for several epochs then gradually climb while training loss continues dropping. This is the classic training curve — the point where val loss is lowest is where we'd want to stop (early stopping). This teaches us that more training isn't always better.

### What Did We Learn?

All three attempts show different training dynamics:

| Attempt | Neurons | Dropout | LR | Behavior |
|---------|---------|---------|-----|----------|
| 1 | 512 | 0.0 | 1e-3 | Fast overfitting — too much capacity |
| 2 | 128 | 0.3 | 3e-4 | Slow, steady — possibly underfitting |
| 3 | 256 | 0.15 | 5e-4 | Classic curve — improvement then overfitting |

The key insight: hyperparameters affect *how* the model trains, not necessarily *how good* it can get. All three likely reach similar best validation losses — they just take different paths.

In practice, you'd pick the configuration that reaches good performance reliably, then use **early stopping** to save the model at its best validation loss.

### Automated Experiments

Manually rewriting the training loop for each configuration is tedious. Let's wrap it in a reusable function and run systematic experiments.

In [ ]:
def train_model(hidden_size, dropout_rate, lr, weight_decay, epochs=40, patience=None, verbose=True):
    """Train a model with given hyperparameters. Returns (model, best_val_loss, best_epoch)."""
    model = NeuralNetwork(embedding_sizes, len(num_columns), hidden_size, dropout_rate).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

    best_val_loss = float('inf')
    best_epoch = 0
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for cats, nums, labels in train_loader:
            cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
            logits = model(cats, nums).squeeze()
            loss = loss_fn(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for cats, nums, labels in val_loader:
                cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
                val_loss += loss_fn(model(cats, nums).squeeze(), labels).item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_epoch = epoch + 1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if verbose:
            print(f"Epoch {epoch+1:2d}: train={avg_train:.4f}, val={avg_val:.4f}")

        if patience and patience_counter >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch+1} (best was epoch {best_epoch})")
            break

    model.load_state_dict(best_state)
    return model, best_val_loss, best_epoch

In [ ]:
# Experiment: Sweep dropout rates
print("=== Dropout Sweep (256 neurons, lr=5e-4) ===")
for dropout in [0.0, 0.1, 0.2, 0.3, 0.5]:
    _, best_loss, best_ep = train_model(
        hidden_size=256, dropout_rate=dropout, lr=5e-4, weight_decay=1e-5,
        epochs=30, patience=10, verbose=False
    )
    print(f"Dropout={dropout:.1f}: best_val_loss={best_loss:.4f} (epoch {best_ep})")

In [ ]:
# Experiment: Sweep network sizes
print("=== Network Size Sweep (dropout=0.15, lr=5e-4) ===")
for size in [64, 128, 256, 512]:
    _, best_loss, best_ep = train_model(
        hidden_size=size, dropout_rate=0.15, lr=5e-4, weight_decay=1e-5,
        epochs=30, patience=10, verbose=False
    )
    print(f"Hidden={size}: best_val_loss={best_loss:.4f} (epoch {best_ep})")

In [ ]:
# Experiment: Sweep learning rates
print("=== Learning Rate Sweep (256 neurons, dropout=0.15) ===")
for lr in [1e-2, 1e-3, 5e-4, 1e-4, 5e-5]:
    _, best_loss, best_ep = train_model(
        hidden_size=256, dropout_rate=0.15, lr=lr, weight_decay=1e-5,
        epochs=30, patience=10, verbose=False
    )
    print(f"LR={lr:.0e}: best_val_loss={best_loss:.4f} (epoch {best_ep})")

### Train the Final Model

Based on the experiments, let's pick a solid configuration and train with early stopping to get our best model.

In [ ]:
model, best_val_loss, best_epoch = train_model(
    hidden_size=128, dropout_rate=0.15, lr=3e-4, weight_decay=1e-5,
    epochs=50, patience=7, verbose=True
)
print(f"\nBest model from epoch {best_epoch} with val_loss={best_val_loss:.4f}")

## A6. Evaluation

The training loop gave us loss numbers, but loss doesn't directly tell us: "is this model useful?" We need metrics that translate to real-world meaning.

First, we need to get our model's predictions on the validation set.

### Getting Predictions

The model outputs **logits** — raw, unbounded numbers. To turn these into predictions:
1. Pass logits through **sigmoid** → probabilities between 0 and 1
2. Apply a **threshold** (0.5) → binary class prediction

In [ ]:
model.eval()
all_logits = []
all_labels = []

with torch.no_grad():
    for cats, nums, labels in val_loader:
        cats, nums, labels = cats.to(device), nums.to(device), labels.to(device)
        logits = model(cats, nums).squeeze()
        all_logits.append(logits)
        all_labels.append(labels)

all_logits = torch.cat(all_logits).cpu()
all_labels = torch.cat(all_labels).cpu()

probabilities = torch.sigmoid(all_logits)
predictions = (probabilities >= 0.5).float()

print(f"Sample predictions: {predictions[:10].int().tolist()}")
print(f"Sample labels:      {all_labels[:10].int().tolist()}")

### Accuracy

The simplest metric: what percentage of predictions were correct?

In [ ]:
accuracy = accuracy_score(all_labels, predictions)
baseline = (y_val == 0).mean()

print(f"Model accuracy: {accuracy:.1%}")
print(f"Baseline (always predict <=50K): {baseline:.1%}")
print(f"Improvement over baseline: +{(accuracy - baseline)*100:.1f} percentage points")

Our model beats the ~75% baseline. The improvement might seem modest at first glance, but remember — we're using `pos_weight` to make the model prioritize catching high earners. This trades some overall accuracy for much better recall on the >50K class. The confusion matrix below will show this trade-off clearly.

Accuracy alone doesn't tell us *where* the model struggles. For that, we need the confusion matrix.

### Confusion Matrix

The confusion matrix breaks accuracy down into four categories:
- **True Negatives (TN)**: Correctly predicted ≤50K
- **False Positives (FP)**: Predicted >50K but actually ≤50K
- **False Negatives (FN)**: Predicted ≤50K but actually >50K
- **True Positives (TP)**: Correctly predicted >50K

In [ ]:
cm = confusion_matrix(all_labels, predictions)
disp = ConfusionMatrixDisplay(cm, display_labels=['<=50K', '>50K'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives:  {tn} (correctly said <=50K)")
print(f"False Positives: {fp} (wrongly said >50K)")
print(f"False Negatives: {fn} (missed >50K earners)")
print(f"True Positives:  {tp} (correctly caught >50K)")
print(f"\nCaught {tp}/{tp+fn} high earners ({tp/(tp+fn)*100:.1f}%)")

### Classification Report

For a complete picture, the classification report gives us per-class **precision**, **recall**, and **F1 score**:

- **Precision**: Of all predictions of a class, how many were correct? ("When you say >50K, how often are you right?")
- **Recall**: Of all actual members of a class, how many did you find? ("Of all high earners, how many did you catch?")
- **F1**: The harmonic mean of precision and recall — a single number that balances both

In [ ]:
print(classification_report(all_labels, predictions, target_names=['<=50K', '>50K']))

### ROC Curve and AUC

One last metric. The ROC curve shows the trade-off between catching true positives and creating false alarms at every possible threshold (not just 0.5).

The **AUC** (Area Under Curve) summarizes this into a single number: if you picked a random high earner and a random low earner, how likely is the model to assign a higher probability to the high earner? AUC=0.5 means random guessing, AUC=1.0 means perfect separation.

In [ ]:
auc = roc_auc_score(all_labels, probabilities)
print(f"AUC-ROC: {auc:.4f}")

RocCurveDisplay.from_predictions(all_labels, probabilities)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.5)')
plt.title(f'ROC Curve (AUC = {auc:.3f})')
plt.legend()
plt.show()

## Reflection

We've built a complete ML pipeline from scratch:

1. **Explored** the data — understood features, found missing values, spotted redundancies
2. **Preprocessed** — cleaned, encoded, scaled, and packaged data for PyTorch
3. **Built** an MLP with embedding layers for categorical features
4. **Trained** with a weighted loss function and tracked overfitting
5. **Tuned** hyperparameters through manual experiments and grid search
6. **Evaluated** with accuracy, confusion matrix, classification report, and AUC-ROC

Questions to consider:
- Which hyperparameters had the biggest impact on performance?
- Where does the model make the most mistakes? Why might that be?
- What would you try next to improve the model?
- We've trained an MLP on tabular data — but is this the best tool for tabular data?

## Reference Materials

- [PyTorch nn.Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)
- [PyTorch DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)
- [PyTorch BCEWithLogitsLoss](https://pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)
- [sklearn Classification Report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [Adult Income Dataset (UCI)](https://archive.ics.uci.edu/dataset/2/adult)

*Updated: 2026_02_17_1200*

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>